# FLAG 2027 - Notebook 01: Data check + Lesson 1 demo

**What this notebook does**
1. Finds the challenge data in your Google Drive (via the shortcut you added).
2. Checks that the training and dev files are complete and correct.
3. Shows you EER in practice (Lesson 1): how well voice-vs-voice and face-vs-face matching works.

**How to run:** Runtime > Run all. No GPU needed (CPU is fine). It takes about 3-5 minutes.

**When done:** copy the printed **SUMMARY** box at the end and paste it to Claude.

In [ ]:
# Cell 1 - connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 - find the challenge zips in Drive and unzip them to Colab's fast local disk
# (Drive is slow with thousands of small files, so we never unzip INTO Drive)
import os, time, shutil, zipfile

BASE = '/content/drive/MyDrive'
LOCAL = '/content/flag_data'          # fast local disk (wiped when the session ends - that's fine)
TARGET = 'train_English_faces.csv'

# 1) find the challenge folder (the shortcut) in My Drive
ZIP_DIR = None
for top in os.listdir(BASE):
    p = os.path.join(BASE, top)
    if os.path.isdir(p) and os.path.exists(os.path.join(p, 'train_set.zip')):
        ZIP_DIR = p
if ZIP_DIR is None:
    raise SystemExit('Could not find a folder with train_set.zip in My Drive. Tell Claude.')
print('Challenge folder:', ZIP_DIR)
for f in sorted(os.listdir(ZIP_DIR)):
    print('   %-60s %8.1f MB' % (f, os.path.getsize(os.path.join(ZIP_DIR, f)) / 1e6))

# 2) copy + unzip each zip locally (skipped if already done in this session)
os.makedirs(LOCAL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    done_flag = os.path.join(LOCAL, z + '.done')
    if os.path.exists(done_flag):
        print(z, 'already unzipped')
        continue
    t0 = time.time()
    shutil.copy(os.path.join(ZIP_DIR, z), '/content/' + z)
    with zipfile.ZipFile('/content/' + z) as zf:
        zf.extractall(LOCAL)
    os.remove('/content/' + z)
    open(done_flag, 'w').close()
    print('%s unzipped in %.0f s' % (z, time.time() - t0))

# 3) locate the data root (folder that contains train_set) inside the unzipped files
found = None
for root, dirs, files in os.walk(LOCAL):
    if TARGET in files:
        found = root
        break
if found is None:
    for root, dirs, files in os.walk(LOCAL):
        d = root.count(os.sep) - LOCAL.count(os.sep)
        if d > 3:
            dirs[:] = []
            continue
        print('  ' * d, os.path.basename(root) + '/', '|', len(files), 'files', files[:4])
    raise SystemExit('Unzipped, but ' + TARGET + ' not found. Copy the output above to Claude.')

p = found
while os.path.basename(p) != 'train_set' and p != LOCAL:
    p = os.path.dirname(p)
DATA_ROOT = os.path.dirname(p) if p != LOCAL else LOCAL
print('\nDATA_ROOT =', DATA_ROOT)
print('Contents:', sorted(os.listdir(DATA_ROOT)))

# 4) index every csv/txt file (skipping raw media folders)
INDEX = []
for root, dirs, files in os.walk(LOCAL):
    dirs[:] = [d for d in dirs if d not in ('English', 'Bangla')]
    for f in files:
        if f.endswith(('.csv', '.txt', '.xlsx', '.json')):
            INDEX.append(os.path.join(root, f))
# the gender meta file sits next to the zips in Drive
if os.path.exists(os.path.join(ZIP_DIR, 'meta_file_train_set.csv')):
    INDEX.append(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'))
print('\nFound %d csv/txt files:' % len(INDEX))
for p in sorted(INDEX):
    print('  ', p)

In [ ]:
# Cell 3 - helper functions
import numpy as np, pandas as pd

def find_one(*parts):
    # return the single indexed path that contains all the given text parts
    hits = [p for p in INDEX if all(x in p for x in parts)]
    if len(hits) != 1:
        print('WARNING: looking for', parts, '-> found', len(hits), hits[:3])
        return None
    return hits[0]

def load_features(path, dim):
    # returns X (n x dim float32) and labels (or None if file has no label column)
    df = pd.read_csv(path, header=None, low_memory=False)
    first = df.iloc[0].astype(str).tolist()
    # drop a header row if there is one (e.g. '0,1,2,...' or text names)
    try:
        vals = [float(v) for v in first[:dim]]
        is_header = vals == list(range(dim))
    except ValueError:
        is_header = True
    if is_header:
        df = df.iloc[1:].reset_index(drop=True)
    X = df.iloc[:, :dim].astype(np.float32).values
    y = df.iloc[:, dim].astype(str).values if df.shape[1] > dim else None
    return X, y, is_header, df.shape

def count_lines(path):
    with open(path) as f:
        return sum(1 for line in f if line.strip())

SUMMARY = []
def note(msg):
    print(msg)
    SUMMARY.append(msg)

In [ ]:
# Cell 4 - check the TRAINING data
tf = find_one('train_English_faces.csv')
tv = find_one('train_English_voices.csv')
Xf, yf, hf, shf = load_features(tf, 4096)
Xv, yv, hv, shv = load_features(tv, 192)

note('TRAIN faces : raw shape %s, header=%s -> X %s' % (shf, hf, Xf.shape))
note('TRAIN voices: raw shape %s, header=%s -> X %s' % (shv, hv, Xv.shape))
note('Labels identical face vs voice rows: %s' % (yf is not None and yv is not None and (yf == yv).all()))

ids, counts = np.unique(yf, return_counts=True)
note('Identities: %d (expected 70) | clips per identity: min %d, median %d, max %d'
     % (len(ids), counts.min(), np.median(counts), counts.max()))
note('Example labels: %s' % list(ids[:5]))
note('Zero rows (failed extraction): faces %d, voices %d'
     % ((np.abs(Xf).sum(1) == 0).sum(), (np.abs(Xv).sum(1) == 0).sum()))
note('NaN values: faces %d, voices %d' % (np.isnan(Xf).sum(), np.isnan(Xv).sum()))

tl = find_one('train_English.txt')
if tl:
    with open(tl) as f:
        head = [next(f).strip() for _ in range(3)]
    note('train_English.txt lines: %d | first line: %s' % (count_lines(tl), head[0]))

In [ ]:
# Cell 5 - find and show the GENDER meta file
# (it was a separate Drive link; it may be in DATA_ROOT or directly in My Drive)
cands = [p for p in INDEX if 'meta' in os.path.basename(p).lower() or 'gender' in os.path.basename(p).lower()]
for f in os.listdir(BASE):
    if ('meta' in f.lower() or 'gender' in f.lower()) and os.path.isfile(os.path.join(BASE, f)):
        cands.append(os.path.join(BASE, f))
# ignore the dev 'gender' track files
cands = [p for p in cands if '_test' not in os.path.basename(p)]
print('Candidates:', cands)

if not cands:
    note('GENDER META: NOT FOUND (add the shortcut to the meta file, then re-run)')
else:
    META = cands[0]
    note('GENDER META: ' + META[len(BASE):])
    if META.endswith('.xlsx'):
        meta_df = pd.read_excel(META)
    else:
        meta_df = pd.read_csv(META, sep=None, engine='python')
    note('meta shape %s, columns %s' % (meta_df.shape, list(meta_df.columns)))
    print(meta_df.head(10))

In [ ]:
# Cell 6 - check the DEV data (we only LOOK at it; rules forbid fitting anything on it)
EXPECTED = {('gender', 'English'): 982, ('gender', 'Bangla'): 1468,
            ('no_gender', 'English'): 1008, ('no_gender', 'Bangla'): 1406}

for (track, lang), exp in EXPECTED.items():
    key = '/' + track + '/'
    lst = find_one(key, lang + '_test.txt')
    fc = find_one(key, lang + '_test_faces.csv')
    vc = find_one(key, lang + '_test_voices.csv')
    if not (lst and fc and vc):
        note('DEV %s/%s: MISSING FILES' % (track, lang))
        continue
    n = count_lines(lst)
    Xdf, _, _, _ = load_features(fc, 4096)
    Xdv, _, _, _ = load_features(vc, 192)
    ok = (n == exp == len(Xdf) == len(Xdv))
    note('DEV %-9s %-7s pairs=%d faces=%d voices=%d expected=%d -> %s'
         % (track, lang, n, len(Xdf), len(Xdv), exp, 'OK' if ok else 'CHECK!'))
    with open(lst) as f:
        print('     first line:', next(f).strip())

## Lesson 1 demo: what EER means

We take the **training** voices and make many pairs:
- **same-person pairs** (two clips of the same speaker)
- **different-person pairs** (clips of two different speakers)

For each pair we compute a **similarity score** (cosine similarity of the two embeddings). A good system gives same-person pairs high scores and different-person pairs low scores.

**EER (Equal Error Rate)** = the error % at the threshold where *wrongly accepted* = *wrongly rejected*. 0% = perfect, 50% = coin toss.

Then we do the same for faces. Note: we **cannot** compare a face embedding with a voice embedding directly - they live in different 'languages' of numbers. Teaching a model to translate between them is the whole challenge.

In [ ]:
# Cell 7 - EER demo on training data
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores)
    fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr))
    return (fpr[i] + fnr[i]) / 2 * 100

def cosine(a, b):
    a = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-9)
    b = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-9)
    return (a * b).sum(1)

rng = np.random.default_rng(0)
N = 5000
by_id = {i: np.where(yf == i)[0] for i in ids}
same_a, same_b, diff_a, diff_b = [], [], [], []
for _ in range(N):
    i = rng.choice(ids)
    a, b = rng.choice(by_id[i], 2, replace=False)
    same_a.append(a); same_b.append(b)
    i, j = rng.choice(ids, 2, replace=False)
    diff_a.append(rng.choice(by_id[i])); diff_b.append(rng.choice(by_id[j]))
A = np.array(same_a + diff_a); B = np.array(same_b + diff_b)
lab = np.array([1] * N + [0] * N)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, (name, X) in zip(axes, [('VOICE vs VOICE', Xv), ('FACE vs FACE', Xf)]):
    s = cosine(X[A], X[B])
    e = eer(lab, s)
    note('Lesson demo - %s EER: %.1f%%' % (name, e))
    ax.hist(s[lab == 1], bins=60, alpha=.6, label='same person')
    ax.hist(s[lab == 0], bins=60, alpha=.6, label='different people')
    ax.set_title('%s  (EER %.1f%%)' % (name, e)); ax.set_xlabel('cosine similarity'); ax.legend()
plt.tight_layout(); plt.show()
print('The more the two colours overlap, the higher the EER.')

In [ ]:
# Cell 8 - save paths for later notebooks + print summary
import json
PROJECT = os.path.join(BASE, 'FLAG2027')          # our own project folder in Drive (results, models)
os.makedirs(PROJECT, exist_ok=True)
with open(os.path.join(PROJECT, 'paths.json'), 'w') as f:
    json.dump({'ZIP_DIR': ZIP_DIR, 'META': META if cands else None,
               'DATA_ROOT_REL': os.path.relpath(DATA_ROOT, LOCAL)}, f, indent=1)

print('=' * 70)
print('SUMMARY  (copy everything between the lines and paste to Claude)')
print('=' * 70)
print('DATA_ROOT:', DATA_ROOT)
for s in SUMMARY:
    print(s)
print('=' * 70)